# 📊 Dashboard de Análisis de Atletas - Intervals.icu

Este cuaderno utiliza el paquete modular `src/` para consultar la API de Intervals.icu, analizar picos de potencia (30 días vs Histórico), calcular métricas de carga (CTL/ATL/TSB) y monitorizar la variabilidad cardíaca (HRV) y peso.

In [6]:
import sys
from pathlib import Path
# Agregar raíz del proyecto al path
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
from config import cargar_roster, DEFAULT_ROSTER_PATH, OUTPUT_DIR
from src import (
    IntervalsClient,
    calcular_picos_potencia,
    generar_tabla_picos_comparativa,
    calcular_metricas_carga,
    resumen_metricas_carga,
    descargar_wellness_atletas,
    resumen_estadisticas_hrv,
    generar_informe_potencias_y_carga,
    generar_informe_wellness_hrv
)

# Inicializar cliente API y cargar roster
client = IntervalsClient()
roster = cargar_roster(DEFAULT_ROSTER_PATH)
atletas = client.get_athletes_list(roster_df=roster, solo_carrera=True)
print(f'✅ Conectado a Intervals.icu con éxito. {len(atletas)} atletas en seguimiento:')
for a in atletas:
    print(f" - {a['athlete_name']} (ID: {a['athlete_id']})")

✅ Conectado a Intervals.icu con éxito. 8 atletas en seguimiento:
 - Adrian Fajardo (ID: i547906)
 - Carlos Garcia (ID: i495562)
 - Daniel Cavia (ID: i701963)
 - Eric Fagundez (ID: i553889)
 - Georgios Bouglas (ID: i554492)
 - Lorenzo Quartucci (ID: i545517)
 - Cesar Macias (ID: i281806)
 - Vojtěch Kmínek (ID: i554486)


## 1. Picos de Potencia: Últimos 30 días vs Histórico
Compara las mejores marcas en 5s, 30s, 1m, 5m, 10m y 20m tanto en potencia absoluta (W) como relativa (W/kg).

In [7]:
peaks_df = calcular_picos_potencia(client, atletas, roster, dias_recientes=30)
tabla_peaks, colores = generar_tabla_picos_comparativa(peaks_df)

# Mostrar tabla estilizada
tabla_peaks.style.apply(lambda _: colores, axis=None)

## 2. Evolución de Carga (CTL / ATL / Ramp Rate)
Calcula el Fitness acumulado (CTL a 42 días) y la Fatiga aguda (ATL a 7 días).

In [8]:
nombres_map = dict(zip(roster['intervals_id'], roster['Name'])) if not roster.empty else {}
metrics_df = calcular_metricas_carga(client, atletas, dias_historia=60, dias_plot=60, nombres_map=nombres_map)
resumen_df = resumen_metricas_carga(metrics_df)
print('📋 Últimos valores de CTL y ATL registrados:')
display(resumen_df[resumen_df['tipo'] == 'FINAL'])

📋 Últimos valores de CTL y ATL registrados:


,athlete_name,fecha,tipo,ctl,atl,tsb,ramp_rate_7d
9,Carlos Garcia,2026-09-14,FINAL,146.6,163.6,-17.0,8.09
19,Cesar Macias,2026-09-14,FINAL,144.7,128.8,15.9,3.29
29,Daniel Cavia,2026-09-14,FINAL,100.5,134.5,-34.0,16.45
39,Georgios Bouglas,2026-09-14,FINAL,84.2,119.3,-35.0,11.94
49,Lorenzo Quartucci,2026-09-14,FINAL,126.7,133.3,-6.5,4.98
59,Vojtěch Kmínek,2026-09-14,FINAL,1.4,0.0,1.4,-0.57


## 3. Bienestar y Variabilidad Cardíaca (HRV)
Descarga y analiza los registros de HRV (RMSSD), peso y descanso.

In [9]:
wellness_df = descargar_wellness_atletas(client, atletas, nombres_map=nombres_map)
stats_hrv = resumen_estadisticas_hrv(wellness_df)
display(stats_hrv)

,athlete_name,dias_con_datos,media_hrv_rmssd,rango_hrv,desv_hrv,ultimo_peso,media_fc_reposo
0,Carlos Garcia,31,NaN,-,NaN,NaN,NaN
1,Cesar Macias,31,86.1,53.0 - 118.0,17.1,NaN,47.9
2,Daniel Cavia,31,NaN,-,NaN,NaN,NaN
3,Eric Fagundez,30,102.5,67.4 - 138.2,18.3,NaN,45.4
4,Georgios Bouglas,31,NaN,-,NaN,NaN,74.0
5,Lorenzo Quartucci,31,NaN,-,NaN,NaN,44.6
6,Vojtěch Kmínek,31,NaN,-,NaN,NaN,NaN


## 4. Generación de Informes en PDF
Crea los documentos vectoriales listos para imprimir o compartir con el cuerpo técnico.

In [10]:
pdf_potencias = generar_informe_potencias_y_carga(peaks_df, tabla_peaks, metrics_df)
print(f'✅ Informe de potencias generado: {pdf_potencias.resolve()}')

if not wellness_df.empty:
    pdf_hrv = generar_informe_wellness_hrv(wellness_df)
    print(f'✅ Informe de HRV generado: {pdf_hrv.resolve()}')

✅ Informe de potencias generado: C:\Users\echav\OneDrive\Documentos\GitHub\procyclingstats\intervals_fit_analytics\output\intervals_informe.pdf
✅ Informe de HRV generado: C:\Users\echav\OneDrive\Documentos\GitHub\procyclingstats\intervals_fit_analytics\output\wellness_evolucion.pdf
